# 18: Retry em camadas (cada parte resolve antes de escalar)

Até aqui, uma exceção dentro de qualquer agente **derrubava a análise inteira**. Se o Gemini demorasse demais
para responder ao Analyzer, o `TimeoutError` subia do nó, atravessava o grafo e a pessoa ficava sem relatório,
mesmo com os dados de mercado já coletados e pagos.

O problema também aparecia em pedaços menores. Numa execução real, o Banco Central devolveu uma resposta que não
era JSON na série do CDI. A chamada não foi repetida: o CDI simplesmente sumiu do relatório. Uma segunda tentativa,
dois segundos depois, quase sempre teria dado certo.

- **SELIC:** a taxa básica de juros da economia, definida pelo Banco Central. É a régua do "dinheiro sem risco".
- **CDI:** a taxa que os bancos cobram entre si por um dia. Anda colada na SELIC e é a referência da renda fixa
  (um CDB "100% do CDI", por exemplo). Sem ela, o relatório perde a comparação com a renda fixa.

A solução é um **retry em camadas**, com um princípio só: **cada agente/parte resolve o retry localmente antes de
escalar pro coordenador**. Quem está mais perto da falha tenta de novo primeiro. Só o que não se resolve ali sobe
para o Supervisor, que decide se manda o agente mais uma vez ou se segue sem ele. Em nenhum caso a análise para.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**Erro temporário x permanente.** Repetir só faz sentido quando a causa pode sumir sozinha:

| Tipo | Exemplos | Repetir? |
|---|---|---|
| Temporário | timeout, conexão caída, resposta que não é JSON, 429 de limite por minuto, 500/503/504 | sim: em segundos pode dar certo |
| Permanente | 400 (pedido inválido), 401 (chave errada), 404 (modelo inexistente), cota zerada, faturamento desligado | não: a mesma chamada falha igual e só gasta |

**Espera crescente (backoff).** Entre uma tentativa e outra o código espera, e a espera dobra a cada vez: 2 s,
depois 4 s. Se o serviço está sobrecarregado, bater de novo na hora só piora. Esperar um pouco dá tempo de ele
se recuperar.

**As três camadas, de baixo para cima:**

1. **Biblioteca do Gemini:** o `langchain-google-genai` já repete sozinho cada chamada ao LLM. O projeto não
    configura nada aqui.
2. **Local:** o nó de cada worker (YFinance, Analyzer, Investigator) roda o agente inteiro até 2 vezes, com 2 s de
    espera. As chamadas externas que o **código** faz sozinho (Banco Central, Brapi, detector de anomalias) repetem
    a própria chamada do mesmo jeito. As tools que o **LLM** chama continuam devolvendo `"Erro ao ..."` como texto:
    quem decide o que fazer com o erro é o agente.
3. **Coordenador:** o que sobrou chega ao Supervisor como um **erro estruturado** no state (`erros` e
    `tentativas`), não como exceção. Erro temporário com o agente abaixo do limite: o Supervisor manda o agente
    mais uma vez. Erro permanente ou limite atingido: a análise segue sem ele e o relatório registra a etapa que
    ficou de fora.

**Por que não a `RetryPolicy` do LangGraph.** O LangGraph tem uma política de retry por nó, mas ela só enxerga a
exceção que **escapa** do nó. Se o nó pega a exceção para devolver um erro estruturado, a política não vê nada. Se
o nó não pega, a política repete e, quando desiste, a exceção derruba o grafo do mesmo jeito. Ela não tem o
terceiro caminho, "seguir sem o agente", que é o que importa aqui. Por isso a camada local é uma função do
projeto (`com_retentativas`, em `shared/retry.py`) chamada dentro do nó.

**A cota zerada que parece instabilidade.** O Gemini devolve o código 429 ("muitas requisições") em dois casos
bem diferentes. Um é o limite por minuto: passou o minuto, passa o erro. O outro é a conta sem faturamento ou com
o crédito pré-pago no fim: esse não passa nunca. Pelo código os dois são iguais, então a classificação também lê
o texto. 429 com `limit: 0`, `billing is not enabled`, `credits` ou `prepay` é permanente. O resto do 429
(inclusive o "You exceeded your current quota ... limit: 15", que é o limite por minuto) é temporário.

## 2. `classificar_erro` com exceções montadas à mão

Os erros do Gemini são montados como o SDK `google-genai` os entrega: o código HTTP e o corpo JSON da resposta.
Nenhuma chamada sai daqui.

In [2]:
import json

import requests
from google.genai import errors as genai_errors

from fluxo_financeiro.shared.retry import RespostaVazia, classificar_erro


def erro_gemini(classe, codigo, mensagem, status):
    """Erro do SDK do Gemini como ele chega da API."""
    return classe(codigo, {"error": {"code": codigo, "message": mensagem, "status": status}})


exemplos = {
    "TimeoutError (rede lenta)": TimeoutError("o servidor demorou demais"),
    "requests.ConnectionError": requests.ConnectionError("conexão recusada"),
    "JSON quebrado (o caso do BCB)": json.JSONDecodeError("Expecting value", "<html>erro</html>", 0),
    "agente sem resposta final": RespostaVazia("o agente terminou sem texto"),
    "Gemini 503 (sobrecarga)": erro_gemini(genai_errors.ServerError, 503, "The model is overloaded.", "UNAVAILABLE"),
    "Gemini 429 limite por minuto": erro_gemini(
        genai_errors.ClientError, 429,
        "You exceeded your current quota. Quota exceeded for metric: requests_per_minute, limit: 15",
        "RESOURCE_EXHAUSTED",
    ),
    "Gemini 429 cota zerada": erro_gemini(
        genai_errors.ClientError, 429,
        "You exceeded your current quota. Quota exceeded for metric: free_tier_requests, limit: 0",
        "RESOURCE_EXHAUSTED",
    ),
    "Gemini 429 sem faturamento": erro_gemini(
        genai_errors.ClientError, 429, "Billing is not enabled for this project.", "RESOURCE_EXHAUSTED",
    ),
    "Gemini 401 (chave inválida)": erro_gemini(genai_errors.ClientError, 401, "API key not valid", "UNAUTHENTICATED"),
    "Gemini 404 (modelo inexistente)": erro_gemini(genai_errors.ClientError, 404, "model not found", "NOT_FOUND"),
    "ValueError (bug no código)": ValueError("valor inesperado"),
}

for nome, erro in exemplos.items():
    print(f"{nome:34} -> {classificar_erro(erro)}")

TimeoutError (rede lenta)          -> transitorio
requests.ConnectionError           -> transitorio
JSON quebrado (o caso do BCB)      -> transitorio
agente sem resposta final          -> transitorio
Gemini 503 (sobrecarga)            -> transitorio
Gemini 429 limite por minuto       -> transitorio
Gemini 429 cota zerada             -> permanente
Gemini 429 sem faturamento         -> permanente
Gemini 401 (chave inválida)        -> permanente
Gemini 404 (modelo inexistente)    -> permanente
ValueError (bug no código)         -> permanente


## 3. `com_retentativas`: repetir só o que vale a pena

Três funções de mentira, com uma espera real curta (`espera=0.1`, então 0,1 s entre tentativas) para a célula
rodar rápido. Na análise de verdade a espera é de 2 s. A linha `[retry]` que aparece no log é o aviso que o
código escreve a cada nova tentativa.

In [3]:
import time

from fluxo_financeiro.shared.retry import com_retentativas


def falha_uma_vez():
    """Simula o BCB: na primeira chamada a resposta não é JSON, na segunda vem certa."""
    falha_uma_vez.chamadas += 1
    if falha_uma_vez.chamadas == 1:
        raise json.JSONDecodeError("Expecting value", "<html>erro</html>", 0)
    return {"CDI": "14,90% a.a."}


falha_uma_vez.chamadas = 0
inicio = time.perf_counter()
resultado = com_retentativas(falha_uma_vez, espera=0.1, contexto="[demo] CDI")
print(f"Temporário que passa: {resultado} | chamadas: {falha_uma_vez.chamadas} | {time.perf_counter() - inicio:.2f} s")


def sempre_permanente():
    sempre_permanente.chamadas += 1
    raise ValueError("chave inválida")


sempre_permanente.chamadas = 0
try:
    com_retentativas(sempre_permanente, espera=0.1, contexto="[demo] permanente")
except ValueError as e:
    print(f"Permanente: levantou {type(e).__name__} na hora | chamadas: {sempre_permanente.chamadas}")


def sempre_timeout():
    sempre_timeout.chamadas += 1
    raise TimeoutError("o servidor demorou demais")


sempre_timeout.chamadas = 0
try:
    com_retentativas(sempre_timeout, espera=0.1, contexto="[demo] esgota")
except TimeoutError as e:
    print(f"Temporário que não passa: levantou {type(e).__name__} depois de {sempre_timeout.chamadas} chamadas")

2026-11-08 22:00:48,342 - fluxo_financeiro - WARNING - [retry] [demo] CDI: JSONDecodeError, tentativa 1 de 2


2026-11-08 22:00:48,444 - fluxo_financeiro - WARNING - [retry] [demo] esgota: TimeoutError, tentativa 1 de 2


Temporário que passa: {'CDI': '14,90% a.a.'} | chamadas: 2 | 0.10 s
Permanente: levantou ValueError na hora | chamadas: 1


Temporário que não passa: levantou TimeoutError depois de 2 chamadas


## 4. O coordenador com estados de erro montados à mão

O nó que esgota as tentativas locais não levanta: ele devolve `erros[agente]` (um `ErroAgente` com o tipo, a
mensagem e quantas rodadas já falharam) e `tentativas[agente]`. O Supervisor lê esses dois campos **antes** dos
guards de etapa. O limite é `MAX_TENTATIVAS = 2` rodadas do nó por etapa: a primeira e mais uma mandada pelo
coordenador.

Os três estados abaixo são resolvidos por guard, sem chamar o LLM:
1. Erro temporário na primeira rodada: o coordenador reenvia o Analyzer (sem tarefa nova e sem somar volta).
2. Erro temporário esgotado (2 rodadas) com o Investigator já concluído no paralelo: o guard encerra em FINISH.
3. Erro permanente na primeira rodada com o Investigator já concluído: o guard encerra em FINISH na hora.

In [4]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.graph.supervisor import supervisor_node
from fluxo_financeiro.shared.retry import MAX_TENTATIVAS, desistiu
from fluxo_financeiro.shared.types import ErroAgente, SinalAnomalia

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: ...", name="YFinance")
tarefa = HumanMessage("Busque notícias recentes sobre a empresa...", name="Supervisor")
erro_msg = HumanMessage("Erro: Analyzer falhou depois de 2 tentativas (transitorio): TimeoutError", name="Analyzer")
diag_msg = HumanMessage("- Causa identificada: queda pontual no 4T", name="Investigator")
sinal = SinalAnomalia(tipo="queda_lucro", descricao="Lucro líquido caiu 45% contra o mesmo trimestre do ano anterior.")


def estado_com_erro(tipo, rodadas, diagnostico=None):
    res = {
        "messages": [pedido, dados, tarefa, erro_msg] + ([diag_msg] if diagnostico else []),
        "loop_count": 2,
        "dados_mercado": dados.text,
        "sinais_anomalia": [sinal],
        "erros": {"Analyzer": ErroAgente(tipo=tipo, mensagem="TimeoutError: o servidor demorou demais", tentativas=rodadas)},
        "tentativas": {"Analyzer": rodadas},
    }
    if diagnostico:
        res["diagnostico"] = diagnostico
    return res


casos = {
    "temporário, 1a rodada falhou": estado_com_erro("transitorio", 1),
    "temporário, 2 rodadas falharam": estado_com_erro("transitorio", 2, diagnostico=diag_msg.text),
    "permanente, 1a rodada falhou": estado_com_erro("permanente", 1, diagnostico=diag_msg.text),
}

print("MAX_TENTATIVAS =", MAX_TENTATIVAS)
for nome, estado in casos.items():
    decisao = supervisor_node(estado)
    print()
    print(f"{nome}: desistiu = {desistiu(estado, 'Analyzer')}")
    print(f"   -> {decisao['next_agent']} | decidido por {decisao['decidido_por']} | loop_count soma {decisao['loop_count']}")
    if decisao.get("messages"):
        print("   tarefa nova:", decisao["messages"][0].text.splitlines()[0])
    else:
        print("   sem tarefa nova: o agente relê a última tarefa do histórico")

2026-11-08 22:00:48,554 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=2


2026-11-08 22:00:48,554 - fluxo_financeiro - WARNING - [supervisor] guard: Analyzer com erro temporário → nova tentativa (2 de 2)


2026-11-08 22:00:48,554 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=2


2026-11-08 22:00:48,555 - fluxo_financeiro - INFO - [supervisor] guard: Investigator concluiu ou desistiu → FINISH


2026-11-08 22:00:48,555 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=2


2026-11-08 22:00:48,556 - fluxo_financeiro - INFO - [supervisor] guard: Investigator concluiu ou desistiu → FINISH


MAX_TENTATIVAS = 2

temporário, 1a rodada falhou: desistiu = False
   -> Analyzer | decidido por guard | loop_count soma 0
   sem tarefa nova: o agente relê a última tarefa do histórico

temporário, 2 rodadas falharam: desistiu = True
   -> FINISH | decidido por guard | loop_count soma 1
   sem tarefa nova: o agente relê a última tarefa do histórico

permanente, 1a rodada falhou: desistiu = True
   -> FINISH | decidido por guard | loop_count soma 1
   sem tarefa nova: o agente relê a última tarefa do histórico


O reenvio devolve `loop_count` 0: ele **não conta volta**. A análise com Investigator usa 3 passagens pelo
Supervisor e o hard stop é 6; se o reenvio contasse, falhas empurrariam o fluxo para o hard stop. Quem
garante que o reenvio termina é o limite por agente, não o `loop_count`.

## 5. Uma análise real da WEGE3 com um Analyzer que falha uma vez

O nó do Analyzer chama `analyzer_agent.invoke(...)`, lido do módulo `fluxo_financeiro.graph.graph` na hora da
chamada. A célula troca esse objeto, só durante a análise, por um que levanta `TimeoutError` na **primeira**
chamada e depois repassa tudo para o agente real. É a falha mais comum na prática: o modelo demorou e a conexão
caiu.

O que se espera: o nó repete o agente sozinho (camada local), a segunda tentativa dá certo e o Supervisor nem
fica sabendo. Como o Analyzer agora roda em passagem única, o agente falso é chamado exatamente 2 vezes (1 falha e 1 sucesso).
Uma análise completa, cerca de 20 a 30 segundos e algumas chamadas ao Gemini.

In [5]:
import importlib
from unittest.mock import patch

from fluxo_financeiro import app

modulo_grafo = importlib.import_module("fluxo_financeiro.graph.graph")


class AnalyzerQueFalhaUmaVez:
    """Levanta TimeoutError na primeira chamada e depois repassa para o agente real."""

    def __init__(self, real):
        self.real = real
        self.chamadas = 0

    def invoke(self, entrada, *args, **kwargs):
        self.chamadas += 1
        if self.chamadas == 1:
            print(">>> Analyzer falso: levantando TimeoutError na chamada 1")
            raise TimeoutError("simulado: o modelo demorou demais para responder")
        print(f">>> Analyzer falso: chamada {self.chamadas}, repassando para o agente real")
        return self.real.invoke(entrada, *args, **kwargs)


falso = AnalyzerQueFalhaUmaVez(modulo_grafo.analyzer_agent)
final = None
with patch.object(modulo_grafo, "analyzer_agent", falso):
    for modo, pedaco in app.stream({"messages": [HumanMessage("Analise a ação WEGE3")]}, stream_mode=["updates", "values"]):
        if modo == "values":
            final = pedaco
            continue
        for no, update in pedaco.items():
            if no == "Supervisor":
                print(f"Supervisor -> {update['next_agent']:12} por {update['decidido_por']:9} | loop_count +{update['loop_count']}")
            elif no in ("YFinance", "Analyzer", "Investigator"):
                print(f"{no:12} erros={update.get('erros')} tentativas={update.get('tentativas')}")

print()
print("O agente do Analyzer foi chamado", falso.chamadas, "vezes")
print("O analyzer_agent do módulo voltou a ser o real:", modulo_grafo.analyzer_agent is falso.real)

2026-11-08 22:00:48,713 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 22:00:50,020 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico, preço e métricas fundamentais para a ação WEGE3.'


2026-11-08 22:00:50,022 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 22:00:50,023 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


Supervisor -> YFinance     por llm       | loop_count +1


2026-11-08 22:00:50,778 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


... (31 outputs left out)

2026-11-08 22:01:14,487 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (5 msgs)


Supervisor -> FINISH       por llm       | loop_count +1


2026-11-08 22:01:16,669 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1750 chars)



O agente do Analyzer foi chamado 2 vezes
O analyzer_agent do módulo voltou a ser o real: True


## 6. O state final

In [6]:
for campo in ("dados_mercado", "noticias", "contexto_setor", "diagnostico"):
    valor = final.get(campo)
    print(f"{campo:15} {'None' if valor is None else f'{len(valor)} caracteres'}")

print()
print("erros:     ", final.get("erros"))
print("tentativas:", final.get("tentativas"))
print("loop_count:", final["loop_count"])
mensagens_de_erro = [m for m in final["messages"] if m.text.startswith("Erro:")]
print("Mensagens 'Erro: ...' no histórico:", len(mensagens_de_erro))
print("Limitações do relatório:", final["relatorio"].limitacoes if final.get("relatorio") else "relatório de emergência")

dados_mercado   3390 caracteres
noticias        1443 caracteres
contexto_setor  1231 caracteres
diagnostico     None

erros:      {'YFinance': None, 'Analyzer': None}
tentativas: {'YFinance': 0, 'Analyzer': 0}
loop_count: 3
Mensagens 'Erro: ...' no histórico: 0
Limitações do relatório: []


## O que observar

- Na classificação, os dois 429 do Gemini têm o mesmo código e destinos opostos: o de limite por minuto (`limit: 15`) é temporário, o de cota zerada (`limit: 0`) e o sem faturamento são permanentes. Quem separa é o texto da mensagem, não o código.
- O JSON quebrado, o caso real do Banco Central, é temporário. No `com_retentativas`, a função que simula o CDI falhou na primeira chamada, deixou uma linha `[retry] [demo] CDI` no log e acertou na segunda, depois de 0,1 s de espera.
- O erro permanente subiu na hora, com uma chamada só e sem linha `[retry]`. O temporário que não passa foi tentado 2 vezes e depois levantou a mesma exceção, para quem chamou decidir.
- No coordenador, o erro temporário com 1 rodada falhada gerou reenvio do Analyzer por guard, com `loop_count` somando 0 e sem tarefa nova. Com as tentativas esgotadas (2 rodadas) ou erro permanente e o outro ramo concluído, o fluxo avançou para FINISH por guard.
- Nenhum dos três estados montados chamou o LLM: todas as decisões saíram com `decidido por: guard`.
- Na WEGE3, o Analyzer falso levantou `TimeoutError` na chamada 1 e o log mostrou `[retry] [node] Analyzer: TimeoutError, tentativa 1 de 2`. Cerca de 2 s depois (a espera local) o nó chamou o agente de novo e deu certo. Tudo isso aconteceu dentro de uma única passagem do nó: o Supervisor não viu erro nenhum e não houve reenvio.
- Depois do sucesso, `erros` ficou `{'Analyzer': None}` e `tentativas` `{'Analyzer': 0}`.
- O agente do Analyzer foi chamado 2 vezes (e não 3): a chamada que falhou e a chamada seguinte que respondeu com as duas seções (`## Notícias da empresa` e `## Contexto do setor`). Os dois campos `noticias` e `contexto_setor` foram gravados na mesma passagem, nenhuma mensagem `Erro: ...` ficou no histórico e o relatório saiu sem limitações.
- Nesta execução a WEGE3 não teve sinal de anomalia e o LLM escolheu FINISH na Etapa 4, então `diagnostico` ficou `None` e o `loop_count` terminou em 3 (o mesmo de uma análise sem falha). A falha local não custou volta.
- Os números e os textos da análise mudam a cada execução. O caminho da falha, não: ele é provocado pelo Analyzer falso, e as decisões do coordenador vêm de guards.